# 08. 베이스라인: ASR + 키워드 규칙 + 상태머신 — **Colab GPU**

```
음성 → ASR (단어별 시각) → 키워드 규칙으로 '일' 추출 → 상태머신 → 질의 시각별 B-S-O → 평가
```

- ASR만 바꿔 두 가지 베이스라인을 낸다: **Whisper large-v3** / **Qwen3-ASR-1.7B (+ ForcedAligner)**
  - Qwen3-ASR은 데이터 합성에 쓴 Qwen3-TTS와 같은 계열이라 합성 음성에 유리할 수 있음 → 결과 해석 때 감안
- 일부러 **단순한** 기준선. 규칙은 템플릿 목록에 맞추지 않고 일반적인 중계 표현 수준으로 만든다
- 판정을 잘못 세도 카운트 언급("투앤원")·타자 소개("원아웃, 타석에는")가 나오면 상태머신이 바로잡는다

**`MODE`** (셀 1): `"pilot"` = 파일럿 v5 20클립, `"main"` = 본 데이터셋 1,300클립. 결과는 `baseline_{ASR}_{pilot_v5|main}/`

**준비**: Drive `kbo-listen/dataset_{pilot_v5|main}/` (06 결과). 받아쓰기는 20클립마다 저장돼서 런타임이 끊겨도 다시 실행하면 이어서 한다.
`kbolisten/`은 Drive for desktop으로 동기화한 폴더(`Othercomputers/*/kbolisten`)에서 자동으로 불러온다

`qwen-asr`과 TTS용 `qwen-tts`는 요구하는 라이브러리 버전이 다를 수 있어서, TTS 노트북과는 **다른 런타임 세션**에서 돌린다

In [8]:
ASR = "qwen3"  # "whisper" 또는 "qwen3"
MODE = "main"    # "pilot"(파일럿 v5 20클립) 또는 "main"(본 데이터셋 1,300클립)
SPLITS = ("dev", "test")  # 평가할 split. 베이스라인 점수에 train은 필요 없음 (받아쓰기 시간·단위 절약)

!pip install -q jiwer
if ASR == "qwen3":
    !pip install -q -U qwen-asr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 722.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.3 MB/s eta 0:00:00
   ━

In [9]:
import glob
import json
import os
import re
import sys
import time
from pathlib import Path

import jiwer
import soundfile as sf
import torch
from google.colab import drive

# kbolisten은 Drive for desktop으로 동기화한 로컬 폴더에서 불러온다 (매번 업로드할 필요 없음)
drive.mount("/content/drive")
_found = glob.glob("/content/drive/Othercomputers/*/kbolisten")
assert _found, "Drive '컴퓨터' 영역에서 kbolisten을 못 찾음 → Drive for desktop 동기화 확인"
sys.path.insert(0, os.path.dirname(_found[0]))

from kbolisten import build as B
from kbolisten.evaluate import evaluate, load_groups
from kbolisten.state_machine import answer_queries

DRIVE = Path("/content/drive/MyDrive/kbo-listen")
DATA = DRIVE / B.DIRS[MODE]["dataset"]  # 06 결과
OUT = DRIVE / f"baseline_{ASR}_{DATA.name.removeprefix('dataset_')}"
OUT.mkdir(exist_ok=True)

queries = json.loads((DATA / "queries.json").read_text(encoding="utf-8"))
gt = json.loads((DATA / "gt.json").read_text(encoding="utf-8"))
groups = load_groups(DATA / "meta.jsonl")
# 고른 split만 남긴다 (groups[cid][0]이 split 이름)
queries = {cid: ts for cid, ts in queries.items() if groups[cid][0] in SPLITS}
gt = {cid: g for cid, g in gt.items() if cid in queries}
print(ASR, "/", DATA.name, len(queries), "클립", sum(len(ts) for ts in queries.values()), "질의")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
qwen3 / dataset_main 300 클립 4687 질의


In [10]:
# 1) 받아쓰기. 어느 ASR이든 transcribe(path) → (전체 텍스트, [{text, start, end}, ...]) 로 맞춘다
#    이미 받아쓴 클립(transcripts.json)은 건너뛴다 → 런타임이 끊겨도 이어서, 규칙만 고칠 때는 모델을 안 올림
TRANSCRIPTS = OUT / "transcripts.json"
transcripts = json.loads(TRANSCRIPTS.read_text(encoding="utf-8")) if TRANSCRIPTS.exists() else {}
todo = [cid for cid in queries if cid not in transcripts]
print(f"저장된 받아쓰기 {len(transcripts)}개, 남은 클립 {len(todo)}개")

if todo:
    if ASR == "whisper":
        from transformers import pipeline

        asr = pipeline(
            "automatic-speech-recognition", model="openai/whisper-large-v3",
            dtype=torch.float16, device="cuda:0",
        )

        def transcribe(path):
            out = asr(
                str(path), return_timestamps="word", chunk_length_s=30, batch_size=2,
                generate_kwargs={"language": "korean", "task": "transcribe"},
            )
            words = []
            for ch in out["chunks"]:
                start, end = ch["timestamp"]
                words.append({"text": ch["text"], "start": start, "end": end if end is not None else start})
            return out["text"], words

    else:
        from qwen_asr import Qwen3ASRModel

        asr = Qwen3ASRModel.from_pretrained(
            "Qwen/Qwen3-ASR-1.7B", dtype=torch.bfloat16, device_map="cuda:0",
            max_inference_batch_size=8, max_new_tokens=1024,  # 1~2분 클립이라 넉넉하게
            forced_aligner="Qwen/Qwen3-ForcedAligner-0.6B",   # 단어별 시각은 정렬 모델이 담당
            forced_aligner_kwargs=dict(dtype=torch.bfloat16, device_map="cuda:0"),
        )

        def flat(items):
            for x in items or []:
                if isinstance(x, (list, tuple)):
                    yield from flat(x)
                else:
                    yield x

        def transcribe(path):
            r = asr.transcribe(audio=str(path), language="Korean", return_time_stamps=True)[0]
            stamps = list(flat(r.time_stamps))
            # 시각 단위가 밀리초로 오는 경우 대비: 마지막 시각이 음성 길이보다 훨씬 크면 초로 바꿈
            scale = 1000 if stamps and stamps[-1].end_time > 1.5 * sf.info(path).duration else 1
            # 정렬 단위는 원문보다 잘게 쪼개져 있다 ("공격" "입니다"). 띄어쓰기는 규칙 셀에서 원문 기준으로 복원
            words = [{"text": s.text, "start": s.start_time / scale, "end": s.end_time / scale} for s in stamps]
            return r.text, words

    t0 = time.perf_counter()
    for k, cid in enumerate(todo, 1):
        text, words = transcribe(DATA / "audio" / f"{cid}.wav")
        transcripts[cid] = {"text": text, "words": words}
        if k % 20 == 0 or k == len(todo):  # 런타임이 끊겨도 여기까지는 남는다
            TRANSCRIPTS.write_text(json.dumps(transcripts, ensure_ascii=False), encoding="utf-8")
            eta = (time.perf_counter() - t0) / k * (len(todo) - k) / 60
            print(f"{k}/{len(todo)} 저장 (남은 예상 {eta:.0f}분)")
    print(f"받아쓰기 {time.perf_counter() - t0:.0f}초")

first = transcripts[next(iter(transcripts))]
print(first["text"][:300])
print(first["words"][:5])

저장된 받아쓰기 300개, 남은 클립 0개
8회말 키움 공격입니다. 스코어로 때 3대 키움 6. 마운드에는 김강현, 노아웃, 타석에는 오선진. 초구 슬라이더 그대로 들어오면서 스트라이크 낮게 빠집니다. 자, 평범한 타구 아웃. 수비가 깔끔했습니다. 투수 교체. 마운드에 정현수 원아웃, 타석에는 어준. 자, 샘루수가 처리합니다. 투수가 잘 막아냈어요. 2아웃 타석에는 송성문 몸쪽 빠집니다. 오늘 컨디션이 좋아 보입니다. 자, 127킬로 슬라이더 왼쪽 파울입니다. 자, 일단 빠집니다. 자, 슬라이더 파울이 됐고요. 128킬로 슬라이더 파울. 142킬로 직구 좋은 코스로 빠져나가
[{'text': '8회말', 'start': 0.72, 'end': 1.2}, {'text': '키움', 'start': 1.76, 'end': 2.0}, {'text': '공격', 'start': 2.0, 'end': 2.32}, {'text': '입니다', 'start': 2.32, 'end': 2.56}, {'text': '스코어로', 'start': 2.96, 'end': 3.04}]


In [11]:
# 3) 받아쓰기 품질: 실제 대본과 비교한 CER (글자 오류율). 띄어쓰기·문장부호는 빼고 비교
def norm(text):
    return re.sub(r"[\s.,!?]", "", text)


scripts = {}
with (DATA / "scripts.jsonl").open(encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        scripts[s["clip_id"]] = " ".join(u["text"] for u in s["utts"])

cer = jiwer.cer([norm(scripts[c]) for c in queries], [norm(transcripts[c]["text"]) for c in queries])
print(f"CER {cer:.2%}")

CER 3.76%


In [12]:
# 4) 키워드 규칙: 위에서부터 우선. 앞 규칙에 걸린 글자는 뒤 규칙이 다시 쓰지 않는다
#    (예: '볼카운트 투앤원'의 '볼'을 볼 판정으로 세지 않도록 카운트 규칙이 먼저 가져감)
NUM = {"노": 0, "원": 1, "투": 2, "쓰리": 3, "쓰": 3, "0": 0, "1": 1, "2": 2, "3": 3}
N = r"(노|원|투|쓰리|쓰|[0-3])"

RULES = [
    # 중계에서 직접 들은 상태 → 보정
    (r"풀\s?카운트", lambda m: ("set_count", {"b": 3, "s": 2})),
    # '원앤투'를 ASR이 '원 엔 투'(Qwen3), '1&2'(Whisper)로 받아쓰기도 한다
    (N + r"\s?(?:앤|엔|&|-)\s?" + N, lambda m: ("set_count", {"b": NUM[m[1]], "s": NUM[m[2]]})),
    (N + r"\s?볼\s?" + N + r"\s?스트라이크", lambda m: ("set_count", {"b": NUM[m[1]], "s": NUM[m[2]]})),
    (r"(원|투|쓰리|[1-3])\s?볼(?!넷)", lambda m: ("set_count", {"b": NUM[m[1]], "s": 0})),  # '투볼' = 2-0 관례
    (N + r"\s?아웃", lambda m: ("set_outs", {"o": NUM[m[1]]})),
    (r"타석에", lambda m: ("new_batter", {})),
    # 타석 종료
    (r"(?:헛\s?스윙|루킹|스트라이크)?[,\s]*(?:삼진|3진)", lambda m: ("in_play", {"outs": 1})),
    (r"볼넷|고의\s?(?:4구|사구)|몸에\s?맞|사구|걸어\s?나", lambda m: ("walk", {})),
    (r"삼중살|트리플\s?플레이", lambda m: ("in_play", {"outs": 3})),
    (r"병살|더블\s?플레이", lambda m: ("in_play", {"outs": 2})),
    # '2루타' '3루 타' '삼루타'처럼 숫자 표기·띄어쓰기가 달라도
    (r"안타|(?:[23]|이|삼)\s?루\s?타|홈런|실책", lambda m: ("in_play", {"outs": 0})),
    (r"(?:잡아냅니다|잡았습니다|처리합니다)?[,.\s]*아웃|잡아냅니다|잡았습니다|처리합니다|희생|야수\s?선택",
     lambda m: ("in_play", {"outs": 1})),
    # 투구 판정
    (r"파울|커트", lambda m: ("foul", {})),
    (r"헛\s?스윙|스트라이크", lambda m: ("strike", {})),
    (r"(?<![땅뜬])볼(?!넷|\s?카운트)|빠집니다|빠졌|골라", lambda m: ("ball", {})),
]
RULES = [(re.compile(p), f) for p, f in RULES]


def realign(text, words):
    """정렬 단위를 받아쓴 원문 위에 순서대로 맞춰서, 원문의 띄어쓰기·문장부호를 되살린 단어 목록을 만든다.
    각 조각 = (직전 단위 끝 ~ 이 단위 끝)까지의 원문, 시각은 이 단위의 시각.
    Qwen3-ForcedAligner는 '공격입니다'를 '공격' '입니다'로 쪼개므로 그냥 이어 붙이면 단어 경계가 틀어진다."""
    out, pos = [], 0
    for w in words:
        unit = w["text"].strip()
        i = text.find(unit, pos) if unit else -1
        if i < 0:  # 원문에서 못 찾은 단위는 건너뜀
            continue
        out.append({"text": text[pos:i + len(unit)], "start": w["start"], "end": w["end"]})
        pos = i + len(unit)
    return out


def words_of(cid):
    t = transcripts[cid]
    return realign(t["text"], t["words"]) if ASR == "qwen3" else t["words"]


def extract_actions(words):
    """단어 목록 → [(시각, 메서드, 인자, 걸린 글자)]. 시각은 걸린 글자가 끝나는 단어의 끝 시각"""
    text, word_end_at = "", []  # word_end_at[i] = 글자 i가 속한 단어의 끝 시각
    for w in words:
        text += w["text"]
        word_end_at += [w["end"]] * len(w["text"])

    used = [False] * len(text)
    actions = []
    for pattern, make in RULES:
        for m in pattern.finditer(text):
            if any(used[m.start():m.end()]):
                continue
            used[m.start():m.end()] = [True] * (m.end() - m.start())
            name, kwargs = make(m)
            actions.append((word_end_at[m.end() - 1], name, kwargs, m.group(0)))
    return sorted(actions, key=lambda a: a[0])


# 첫 클립: 복원된 단어 몇 개와 뽑힌 일 확인
cid = next(iter(transcripts))
print(words_of(cid)[:6])
for t, name, kwargs, hit in extract_actions(words_of(cid))[:15]:
    print(f"{t:6.2f}  {name:<10} {kwargs}  ← '{hit.strip()}'")

[{'text': '8회말', 'start': 0.72, 'end': 1.2}, {'text': ' 키움', 'start': 1.76, 'end': 2.0}, {'text': ' 공격', 'start': 2.0, 'end': 2.32}, {'text': '입니다', 'start': 2.32, 'end': 2.56}, {'text': '. 스코어로', 'start': 2.96, 'end': 3.04}, {'text': ' 때', 'start': 3.04, 'end': 3.28}]
  7.36  set_outs   {'o': 0}  ← '노아웃'
  8.24  new_batter {}  ← '타석에'
 12.48  strike     {}  ← '스트라이크'
 14.24  ball       {}  ← '빠집니다'
 17.36  in_play    {'outs': 1}  ← '아웃'
 24.48  set_outs   {'o': 1}  ← '원아웃'
 25.84  new_batter {}  ← '타석에'
 28.16  in_play    {'outs': 1}  ← '처리합니다'
 31.68  set_outs   {'o': 2}  ← '2아웃'
 32.80  new_batter {}  ← '타석에'
 34.56  ball       {}  ← '빠집니다'
 41.76  foul       {}  ← '파울'
 44.40  ball       {}  ← '빠집니다'
 46.96  foul       {}  ← '파울'
 50.48  foul       {}  ← '파울'


In [13]:
# 5) 예측 + 평가
pred = {}
for cid, ts in queries.items():
    actions = [(t, name, kwargs) for t, name, kwargs, _ in extract_actions(words_of(cid))]
    pred[cid] = answer_queries(actions, ts)

(OUT / "pred.json").write_text(json.dumps(pred, ensure_ascii=False), encoding="utf-8")
scores = evaluate(gt, pred, groups)
(OUT / "score.json").write_text(json.dumps({"cer": cer, **scores}, ensure_ascii=False, indent=1), encoding="utf-8")

print(f"{ASR} / {DATA.name} / CER {cer:.2%}")
print(f"{'group':<12} {'n':>5} {'exact':>7} {'ball':>7} {'strike':>7} {'out':>7}")
for name, s in scores.items():
    print(f"{name:<12} {s['n']:>5} {s['exact']:>7.2%} {s['b']:>7.2%} {s['s']:>7.2%} {s['o']:>7.2%}")

qwen3 / dataset_main / CER 3.76%
group            n   exact    ball  strike     out
all           4687  74.95%  85.56%  87.80%  92.92%
dev           1534  77.51%  87.55%  88.40%  93.61%
dev/easy       772  88.99%  96.11%  93.52%  96.24%
dev/hard       762  65.88%  78.87%  83.20%  90.94%
test          3153  73.71%  84.59%  87.50%  92.58%
test/easy     1514  83.36%  93.20%  90.62%  95.18%
test/hard     1639  64.80%  76.63%  84.62%  90.18%


In [14]:
# 6) 오답 분석: 틀린 질의 바로 앞에 무슨 말이 있었고 무엇을 뽑았는지
def show_errors(cid, limit=8):
    words = words_of(cid)
    actions = extract_actions(words)
    shown = 0
    for g, p in zip(gt[cid], pred[cid]):
        if (g["b"], g["s"], g["o"]) == (p["b"], p["s"], p["o"]):
            continue
        heard = "".join(w["text"] for w in words if g["t"] - 4 <= w["end"] <= g["t"])
        took = [f"{n}{kw or ''}" for t, n, kw, _ in actions if g["t"] - 4 <= t <= g["t"]]
        print(f"t={g['t']:6.2f} 정답 {g['b']}-{g['s']} {g['o']}아웃 / 예측 {p['b']}-{p['s']} {p['o']}아웃")
        print(f"         들은 말: {heard.strip()}")
        print(f"         뽑은 일: {took}")
        shown += 1
        if shown == limit:
            break


worst = min(queries, key=lambda c: evaluate({c: gt[c]}, pred)["all"]["exact"])
print("가장 많이 틀린 클립:", worst, groups[worst])
show_errors(worst)

가장 많이 틀린 클립: 20260724KTLT02026_06B ['test', 'test/hard']
t= 12.00 정답 1-0 0아웃 / 예측 0-0 0아웃
         들은 말: , 노아웃 타석에는 윤동이 빠르게 들어옵니다
         뽑은 일: ["set_outs{'o': 0}", 'new_batter']
t= 16.18 정답 1-1 0아웃 / 예측 0-0 0아웃
         들은 말: . 네, 그렇습니다. 뒤쪽으로 걷어냅니다
         뽑은 일: []
t= 20.60 정답 1-2 0아웃 / 예측 0-1 0아웃
         들은 말: . 네, 그렇습니다. 가운데로 쏘아지며 스트라이크
         뽑은 일: ['strike']
t= 25.84 정답 2-2 0아웃 / 예측 0-2 0아웃
         들은 말: 건 투수를 힘들게 하죠. 가슴 쪽으로 빠져나갑니다
         뽑은 일: []
t= 42.93 정답 0-1 1아웃 / 예측 0-0 1아웃
         들은 말: 타석에는 손성빈 바깥쪽에 나가면서 허스윙
         뽑은 일: ['new_batter']
t= 45.23 정답 1-1 1아웃 / 예측 0-0 1아웃
         들은 말: 바깥쪽에 나가면서 허스윙 바깥쪽으로 빠져나갑니다
         뽑은 일: []
t= 53.59 정답 2-2 1아웃 / 예측 1-2 1아웃
         들은 말: 이번 타석이 정말 중요하죠. 곤란해집니다
         뽑은 일: []
t= 57.05 정답 2-2 1아웃 / 예측 1-2 1아웃
         들은 말: 해집니다. 149 킬로 직구 파울
         뽑은 일: ['foul']
